# 01 - Ablation tables

Turns the raw per-run result files of the Leave-One-Feature-Out ablation
study into a small set of tidy CSV tables: which feature, when removed,
hurts the reconstruction of the others the most.

**Inputs**

* `results_CAISE/<log>/<pattern>_V2_RESULTS_END.csv` - baseline model (all attributes), 10 runs
* `results_CAISE/ablation/<log>/ablation_<feature>/<pattern>_V2_RESULTS_END.csv` - ablated variants, 10 runs
* `data/dataset_features.json` - attribute list and type of each log

**Outputs** (written in `analysis/tables/`)

| file | content |
|---|---|
| `runs_tidy.csv` | one row per (log, config, pattern, run, attribute): the raw measurements |
| `baseline_reference.csv` | baseline mean/std per (log, pattern, attribute) |
| `delta_per_attribute.csv` | Delta_i,j of eq. (2.1) + a simple t-test |
| `delta_aggregated.csv` | Delta_cat / Delta_num of eq. (2.2), per (log, ablated feature, pattern) |
| `ranking.csv` | final importance ranking per log, averaged over the 5 patterns |


In [1]:
import json
import os
import re
import warnings

import numpy as np
import pandas as pd
from scipy import stats

In [2]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

In [3]:
root_path = os.getcwd()
root = os.path.dirname(root_path)
print(root_path, root)

/home/danbi/Projects/SANAGRAPH/analysis /home/danbi/Projects/SANAGRAPH


In [4]:
result_dir = os.path.join(root, "results_CAISE")
ablation_dir = os.path.join(result_dir, "ablation")
tables_dir = os.path.join(root, "analysis", "tables")
os.makedirs(tables_dir, exist_ok=True)
print(result_dir, ablation_dir, tables_dir)

/home/danbi/Projects/SANAGRAPH/results_CAISE /home/danbi/Projects/SANAGRAPH/results_CAISE/ablation /home/danbi/Projects/SANAGRAPH/analysis/tables


In [5]:
logs = ["bpi_2012_CZ", "bpi_2013_CZ", "sp2020_CZ", "BPI20_RequestForPayment_CZ"]
patterns = ["odd", "even", "window", "random", "attr_level"]
n_runs = 10

In [6]:
with open(os.path.join(root, "data", "dataset_features.json")) as f:
    dataset_features = json.load(f)

In [7]:
def convert_feature_name(feature):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", feature)

In [8]:
def attributes_of(log):
    info = dataset_features[log]
    return {**{a: "categorical" for a in info["categorical"]},
            **{a: "numerical" for a in info["numerical"]}}

In [9]:
print(f"root      : {root}")
print(f"logs      : {len(logs)}")
print(f"variants  : {sum(len(attributes_of(l)) for l in logs)} (one per attribute of each log)")

root      : /home/danbi/Projects/SANAGRAPH
logs      : 4
variants  : 33 (one per attribute of each log)


## 1. Raw runs

Each log has one `baseline` configuration (all attributes) and one
`ablate::<attribute>` configuration per attribute. Every configuration was
trained `N_RUNS` times and evaluated under the five missingness patterns, so
each result file has exactly 10 rows, one per random initialisation.

In [10]:
def config_directories(log):
    """(config, excluded_feature, directory) for every configuration of a log."""
    out = [("baseline", None, os.path.join(result_dir, log))]
    for feature in attributes_of(log):
        out.append((f"ablate::{feature}", feature,
                    os.path.join(ablation_dir, log, f"ablation_{convert_feature_name(feature)}")))
    return out


frames, missing_files = [], []
for log in logs:
    for config, excluded, directory in config_directories(log):
        for pattern in patterns:
            path = os.path.join(directory, f"{pattern}_V2_RESULTS_END.csv")
            if not os.path.exists(path):
                missing_files.append(os.path.relpath(path, root))
                continue
            df = pd.read_csv(path)
            df.insert(0, "run", range(len(df)))
            df.insert(0, "pattern", pattern)
            df.insert(0, "excluded_feature", excluded)
            df.insert(0, "config", config)
            df.insert(0, "log", log)
            frames.append(df)

runs_wide = pd.concat(frames, ignore_index=True)

assert not missing_files, f"missing result files: {missing_files}"
counts = runs_wide.groupby(["log", "config", "pattern"]).size()
assert (counts == n_runs).all(), counts[counts != n_runs]

print(f"{runs_wide.shape[0]} rows = "
      f"{runs_wide.groupby(['log', 'config']).ngroups} configurations "
      f"x {len(patterns)} patterns x {n_runs} runs")
runs_wide.head()

1850 rows = 37 configurations x 5 patterns x 10 runs


,log,config,excluded_feature,pattern,run,Activity_acc,org:resource_acc,concept:name_acc,lifecycle:transition_acc,time:timestamp_mae,(case) AMOUNT_REQ_mae,AVG_total_loss,MAE_time:timestamp,MAE_(case) AMOUNT_REQ,impact_acc,org:group_acc,org:role_acc,organization country_acc,organization involved_acc,product_acc,resource country_acc,REPAIR_IN_TIME_5D_acc,DEVICETYPE_acc,case:Project_acc,case:Task_acc,case:OrganizationalEntity_acc,case:Activity_acc,case:RfpNumber_acc,case:RequestedAmount_mae,MAE_case:RequestedAmount
0,bpi_2012_CZ,baseline,None,odd,0,0.954275,0.778998,0.950486,0.990183,0.010063,0.061131,1.761607,0.010054,0.060945,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,bpi_2012_CZ,baseline,None,odd,1,0.950831,0.780332,0.954318,0.988720,0.022088,0.022533,1.713222,0.021993,0.022264,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bpi_2012_CZ,baseline,None,odd,2,0.955825,0.777362,0.954620,0.990011,0.007492,0.029271,1.718253,0.007513,0.028891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bpi_2012_CZ,baseline,None,odd,3,0.948592,0.761646,0.953199,0.989882,0.007986,0.032011,1.770121,0.008021,0.031804,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,bpi_2012_CZ,baseline,None,odd,4,0.954275,0.778610,0.948506,0.988074,0.020917,0.021581,1.724486,0.020922,0.021425,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Tidy format

The wide format has one column per attribute and is sparse, since each log
has its own attributes. We reshape it into one row per single measurement,
keeping only the two metrics used for the analysis: accuracy for categorical
attributes, and MAE for numerical ones (the exact MAE over all rebuilt
nodes, column `MAE_<a>`, not the per-batch loss mean `<a>_mae`).

In [11]:
records = []
for log in logs:
    kinds = attributes_of(log)
    block = runs_wide[runs_wide.log == log]
    for attribute, kind in kinds.items():
        column = f"{attribute}_acc" if kind == "categorical" else f"MAE_{attribute}"
        metric = "accuracy" if kind == "categorical" else "mae"
        sub = block[block[column].notna()]
        records.append(pd.DataFrame({
            "log": sub.log, "config": sub.config, "excluded_feature": sub.excluded_feature,
            "pattern": sub.pattern, "run": sub.run, "attribute": attribute,
            "attribute_kind": kind, "metric": metric, "value": sub[column].astype(float),
        }))

runs_tidy = pd.concat(records, ignore_index=True).sort_values(
    ["log", "config", "pattern", "attribute", "run"]).reset_index(drop=True)
runs_tidy.to_csv(os.path.join(tables_dir, "runs_tidy.csv"), index=False)

print(f"runs_tidy.csv -> {len(runs_tidy)} rows")
runs_tidy.head()

runs_tidy.csv -> 15250 rows


,log,config,excluded_feature,pattern,run,attribute,attribute_kind,metric,value
0,BPI20_RequestForPayment_CZ,ablate::Activity,Activity,attr_level,0,case:Activity,categorical,accuracy,0.984754
1,BPI20_RequestForPayment_CZ,ablate::Activity,Activity,attr_level,1,case:Activity,categorical,accuracy,0.974136
2,BPI20_RequestForPayment_CZ,ablate::Activity,Activity,attr_level,2,case:Activity,categorical,accuracy,0.974952
3,BPI20_RequestForPayment_CZ,ablate::Activity,Activity,attr_level,3,case:Activity,categorical,accuracy,0.973863
4,BPI20_RequestForPayment_CZ,ablate::Activity,Activity,attr_level,4,case:Activity,categorical,accuracy,0.982576


## 3. Baseline reference

Mean and standard deviation of the baseline over the 10 runs: the reference
term of every Delta, and what tells whether a Delta is large or small
relative to what the baseline already achieves.

In [12]:
baseline_reference = (runs_tidy[runs_tidy.config == "baseline"]
                      .groupby(["log", "pattern", "attribute", "attribute_kind", "metric"])["value"]
                      .agg(mean="mean", std=lambda x: x.std(ddof=1))
                      .reset_index())
baseline_reference.to_csv(os.path.join(tables_dir, "baseline_reference.csv"), index=False)

baseline_reference.pivot_table(index=["log", "attribute", "metric"],
                               columns="pattern", values="mean").round(4)

pattern                                                        attr_level    even     odd  random  window
log                        attribute                 metric                                              
BPI20_RequestForPayment_CZ Activity                  accuracy      0.9195  0.9667  0.9663  0.8138  0.9507
                           case:Activity             accuracy      0.9702  1.0000  1.0000  0.9781  1.0000
                           case:OrganizationalEntity accuracy      0.8330  0.9464  0.9423  0.8479  0.9377
                           case:Project              accuracy      0.8119  0.9364  0.9333  0.8345  0.9262
                           case:RequestedAmount      mae           0.2948  0.0764  0.0629  0.2672  0.1074
                           case:RfpNumber            accuracy      0.0380  0.0447  0.0317  0.0413  0.0400
                           case:Task                 accuracy      0.7585  0.7556  0.7522  0.7540  0.7539
                           org:resource              accuracy      0.9913  0.9998  0.9994  0.9178  0.9983
                           org:role                  accuracy      0.9674  0.9923  0.9866  0.8791  0.9909
                           time:timestamp            mae           0.0076  0.0068  0.0071  0.0076  0.0075
bpi_2012_CZ                (case) AMOUNT_REQ         mae           0.0971  0.0257  0.0257  0.0839  0.0229
                           Activity                  accuracy      0.8954  0.9232  0.9519  0.7723  0.8937
                           concept:name              accuracy      0.9002  0.9239  0.9522  0.7927  0.8883
                           lifecycle:transition      accuracy      0.9597  0.9893  0.9892  0.8903  0.9567
                           org:resource              accuracy      0.6573  0.7940  0.7736  0.6417  0.7176
                           time:timestamp            mae           0.0288  0.0119  0.0125  0.0177  0.0121
bpi_2013_CZ                Activity                  accuracy      0.7536  0.6877  0.6660  0.6214  0.6454
                           concept:name              accuracy      0.9005  0.7452  0.8456  0.7486  0.7854
                           impact                    accuracy      0.9132  1.0000  1.0000  0.9446  1.0000
                           lifecycle:transition      accuracy      0.8184  0.7048  0.6929  0.6499  0.6974
                           org:group                 accuracy      0.8745  0.9478  0.9391  0.8755  0.9498
                           org:resource              accuracy      0.1043  0.1619  0.1687  0.1151  0.1547
                           org:role                  accuracy      0.5914  0.7629  0.7632  0.6753  0.7318
                           organization country      accuracy      0.8907  0.9700  0.9766  0.9159  0.9635
                           organization involved     accuracy      0.2730  0.4153  0.3985  0.3172  0.3856
                           product                   accuracy      0.3592  0.4356  0.4352  0.3807  0.4242
                           resource country          accuracy      0.7726  0.8688  0.8645  0.8149  0.8376
                           time:timestamp            mae           0.0206  0.0197  0.0179  0.0212  0.0182
sp2020_CZ                  Activity                  accuracy      0.6462  0.8491  0.8089  0.6452  0.7388
                           DEVICETYPE                accuracy      0.9390  0.9979  0.9980  0.9359  0.9974
                           REPAIR_IN_TIME_5D         accuracy      0.9870  1.0000  1.0000  0.9830  1.0000
                           org:resource              accuracy      0.9587  0.9998  0.9998  0.9691  0.9997
                           time:timestamp            mae           0.0202  0.0129  0.0160  0.0163  0.0154

## 4. Per-attribute Delta

For a variant obtained by removing attribute `a_i`, and for each surviving
attribute `a_j`, eq. (2.1) of the thesis defines

Delta_i,j = mean(baseline, a_j) - mean(variant, a_j) a_j categorical (accuracy)

Delta_i,j = mean(variant, a_j) - mean(baseline, a_j) a_j numerical (MAE)


so that a **positive Delta always means degradation** caused by removing `a_i`.

Since both means come from 10 runs, each Delta is checked with a simple
two-sample t-test between the two sets of 10 runs (same idea as
`main_notebook_T_Test.ipynb`, but comparing two distributions here instead
of one distribution against a single literature value). `significant` just
flags `p_value < 0.05`: a plain heuristic to separate a real effect from
run-to-run noise, not a rigorous multiple-testing correction.

In [13]:
# (log, config, pattern, attribute) -> the 10 measurements, ordered by run
samples = {k: g.sort_values("run").value.to_numpy()
           for k, g in runs_tidy.groupby(["log", "config", "pattern", "attribute"])}

rows = []
for log in logs:
    kinds = attributes_of(log)
    configs = [c for c in runs_tidy[runs_tidy.log == log].config.unique() if c != "baseline"]
    for config in sorted(configs):
        excluded = config.split("::", 1)[1]
        for pattern in patterns:
            for attribute, kind in kinds.items():
                if attribute == excluded:
                    continue
                b = samples[(log, "baseline", pattern, attribute)]
                v = samples[(log, config, pattern, attribute)]
                b_mean, v_mean = b.mean(), v.mean()
                if kind == "categorical":
                    delta = b_mean - v_mean
                else:
                    delta = v_mean - b_mean
                _, p_value = stats.ttest_ind(b, v, equal_var=False)
                rows.append({
                    "log": log, "excluded_feature": excluded,
                    "structural": excluded == "Activity",
                    "pattern": pattern, "target": attribute, "target_kind": kind,
                    "baseline_mean": b_mean, "variant_mean": v_mean,
                    "delta": delta, "delta_pct": 100 * delta / b_mean,
                    "p_value": p_value, "significant": p_value < 0.05,
                })

delta_per_attribute = pd.DataFrame(rows)
delta_per_attribute.to_csv(os.path.join(tables_dir, "delta_per_attribute.csv"), index=False)

print(f"delta_per_attribute.csv -> {len(delta_per_attribute)} comparisons "
      f"({delta_per_attribute.significant.sum()} with p < 0.05)")

delta_per_attribute[(delta_per_attribute.log == "sp2020_CZ") & (delta_per_attribute.pattern == "random")][["excluded_feature", "target", "baseline_mean", "variant_mean", "delta", "p_value", "significant"]].round(5)

/home/danbi/Projects/SANAGRAPH/venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


delta_per_attribute.csv -> 1360 comparisons (286 with p < 0.05)


,excluded_feature,target,baseline_mean,variant_mean,delta,p_value,significant
822,Activity,REPAIR_IN_TIME_5D,0.98296,0.98750,-0.00453,0.00868,True
823,Activity,DEVICETYPE,0.93588,0.94680,-0.01092,0.00005,True
824,Activity,org:resource,0.96906,0.95776,0.01131,0.00000,True
825,Activity,time:timestamp,0.01633,0.01641,0.00008,0.87309,False
842,DEVICETYPE,Activity,0.64516,0.65246,-0.00731,0.00703,True
843,DEVICETYPE,REPAIR_IN_TIME_5D,0.98296,0.98405,-0.00109,0.57474,False
844,DEVICETYPE,org:resource,0.96906,0.96924,-0.00017,0.86087,False
845,DEVICETYPE,time:timestamp,0.01633,0.01544,-0.00089,0.11480,False
862,REPAIR_IN_TIME_5D,Activity,0.64516,0.64971,-0.00455,0.13485,False
863,REPAIR_IN_TIME_5D,DEVICETYPE,0.93588,0.93950,-0.00362,0.08521,False


In [ ]:
import numpy as np

ALPHA = 0.05

mask = delta_per_attribute.p_value.notna()
p = delta_per_attribute.loc[mask, "p_value"].to_numpy()
order = np.argsort(p)
n = len(p)

below = p[order] <= ALPHA * np.arange(1, n + 1) / n
k = np.max(np.nonzero(below)[0]) + 1 if below.any() else 0

reject = np.zeros(n, dtype=bool)
reject[order[:k]] = True

delta_per_attribute["significant_bh"] = False
delta_per_attribute.loc[mask, "significant_bh"] = reject
delta_per_attribute.to_csv(os.path.join(tables_dir, "delta_per_attribute.csv"), index=False)

print(f"tests performed           : {n}")
print(f"expected by chance (5%)   : {ALPHA * n:.0f}")
print(f"raw p < 0.05              : {int(delta_per_attribute.significant.sum())}")
print(f"Benjamini-Hochberg FDR 5% : {int(delta_per_attribute.significant_bh.sum())}")
print(f"Bonferroni                : {int((p < ALPHA / n).sum())}")

survivors = (delta_per_attribute[delta_per_attribute.significant_bh]
             .groupby(["log", "excluded_feature", "structural"], as_index=False)
             .agg(n_significant_bh=("delta", "size"), mean_delta=("delta", "mean"))
             .sort_values("n_significant_bh", ascending=False))
survivors.to_csv(os.path.join(tables_dir, "bh_survivors.csv"), index=False)
survivors.head(14).round(4)

tests performed           : 1264
expected by chance (5%)   : 63
raw p < 0.05              : 286
Benjamini-Hochberg FDR 5% : 111
Bonferroni                : 44


,log,excluded_feature,structural,n_significant_bh,mean_delta
8,bpi_2012_CZ,Activity,True,18,0.0783
12,bpi_2013_CZ,Activity,True,14,0.0933
7,bpi_2012_CZ,(case) AMOUNT_REQ,False,13,-0.0115
3,BPI20_RequestForPayment_CZ,case:RfpNumber,False,7,-0.0582
6,BPI20_RequestForPayment_CZ,time:timestamp,False,6,-0.0477
21,sp2020_CZ,org:resource,False,5,0.0329
1,BPI20_RequestForPayment_CZ,case:OrganizationalEntity,False,5,-0.0134
17,bpi_2013_CZ,org:role,False,5,0.0720
19,sp2020_CZ,Activity,True,4,-0.0021
16,bpi_2013_CZ,org:resource,False,4,0.0355


## 5. Aggregated Delta

Eq. (2.2) averages the Delta over the surviving attributes, keeping
categorical and numerical separate since accuracy and MAE are not on the
same scale. As established in section 2.2.3, the `Activity` variant is kept
apart (`structural = True`): removing it disconnects the graph rather than
subtracting information at constant topology, so its Delta is not
comparable with the others.

In [15]:
delta_aggregated = (delta_per_attribute
                    .groupby(["log", "excluded_feature", "structural", "pattern"], as_index=False)
                    .apply(lambda g: pd.Series({
                        "delta_cat": g.loc[g.target_kind == "categorical", "delta"].mean(),
                        "delta_num": g.loc[g.target_kind == "numerical", "delta"].mean(),
                        "n_significant": g.significant.sum(),
                        "n_targets": len(g),
                    }), include_groups=False))
delta_aggregated.to_csv(os.path.join(tables_dir, "delta_aggregated.csv"), index=False)

delta_aggregated[~delta_aggregated.structural].pivot_table(
    index=["log", "excluded_feature"], columns="pattern", values="delta_cat"
)[patterns].round(4)

pattern                                                  odd    even  window  random  attr_level
log                        excluded_feature                                                     
BPI20_RequestForPayment_CZ case:Activity             -0.0009  0.0009 -0.0010  0.0023      0.0032
                           case:OrganizationalEntity -0.0029 -0.0001 -0.0019  0.0023      0.0011
                           case:Project              -0.0003  0.0011  0.0007  0.0003      0.0003
                           case:RequestedAmount       0.0018  0.0009 -0.0010  0.0035     -0.0007
                           case:RfpNumber            -0.0038 -0.0014 -0.0047 -0.0059     -0.0058
                           case:Task                  0.0005  0.0001 -0.0019 -0.0024      0.0029
                           org:resource              -0.0003  0.0022 -0.0004 -0.0035      0.0077
                           org:role                  -0.0014 -0.0006 -0.0012 -0.0084      0.0030
                           time:timestamp             0.0002  0.0009 -0.0001  0.0012     -0.0008
bpi_2012_CZ                (case) AMOUNT_REQ         -0.0036 -0.0035 -0.0082 -0.0125     -0.0181
                           concept:name               0.0021  0.0018  0.0005 -0.0006      0.0328
                           lifecycle:transition       0.0008  0.0008  0.0049  0.0034      0.0173
                           org:resource               0.0034  0.0040  0.0077  0.0041      0.0052
                           time:timestamp            -0.0020 -0.0021 -0.0042 -0.0047     -0.0046
bpi_2013_CZ                concept:name               0.0094  0.0097  0.0061  0.0069      0.0184
                           impact                     0.0156  0.0145  0.0111  0.0119      0.0163
                           lifecycle:transition       0.0108  0.0147  0.0068  0.0094      0.0258
                           org:group                  0.0043  0.0079  0.0074  0.0053      0.0088
                           org:resource               0.0124  0.0162  0.0053  0.0103      0.0177
                           org:role                   0.0167  0.0183  0.0168  0.0192      0.0146
                           organization country       0.0073  0.0097  0.0080  0.0078      0.0068
                           organization involved      0.0068  0.0051  0.0023  0.0028      0.0028
                           product                    0.0050  0.0082  0.0020  0.0025      0.0028
                           resource country           0.0152  0.0173  0.0130  0.0161      0.0191
                           time:timestamp             0.0238  0.0216  0.0226  0.0192      0.0206
sp2020_CZ                  DEVICETYPE                -0.0033 -0.0037 -0.0026 -0.0029     -0.0036
                           REPAIR_IN_TIME_5D         -0.0005 -0.0016 -0.0008 -0.0027     -0.0029
                           org:resource               0.0142  0.0054  0.0152  0.0068      0.0083
                           time:timestamp            -0.0008 -0.0011 -0.0010 -0.0023     -0.0021

## 6. Importance ranking

A single number per attribute, averaging the aggregated Delta over the five
missingness patterns: this is the ranking of "which feature matters most"
for each log.

In [16]:
ranking = (delta_aggregated
           .groupby(["log", "excluded_feature", "structural"], as_index=False)
           .agg(delta_cat=("delta_cat", "mean"),
                delta_num=("delta_num", "mean"),
                n_significant=("n_significant", "sum"),
                n_comparisons=("n_targets", "sum")))

ranking["rank"] = (ranking[~ranking.structural]
                   .groupby("log")["delta_cat"].rank(ascending=False, method="min")).astype("Int64")
ranking = ranking.sort_values(["log", "structural", "rank"]).reset_index(drop=True)
ranking.to_csv(os.path.join(tables_dir, "ranking.csv"), index=False)

for log in logs:
    print(f"\n=== {log} ===")
    print(ranking[ranking.log == log][
        ["rank", "excluded_feature", "structural", "delta_cat", "delta_num",
         "n_significant", "n_comparisons"]
    ].to_string(index=False, float_format=lambda x: f"{x:.4f}"))


=== bpi_2012_CZ ===
 rank     excluded_feature  structural  delta_cat  delta_num  n_significant  n_comparisons
    1         concept:name       False     0.0073    -0.0028         5.0000        25.0000
    2 lifecycle:transition       False     0.0055     0.0004         5.0000        25.0000
    3         org:resource       False     0.0049     0.0007         6.0000        25.0000
    4       time:timestamp       False    -0.0035    -0.0014         5.0000        25.0000
    5    (case) AMOUNT_REQ       False    -0.0092    -0.0055        23.0000        25.0000
 <NA>             Activity        True     0.0958    -0.0033        20.0000        25.0000

=== bpi_2013_CZ ===
 rank      excluded_feature  structural  delta_cat  delta_num  n_significant  n_comparisons
    1        time:timestamp       False     0.0216        NaN        14.0000        55.0000
    2              org:role       False     0.0171     0.0024        12.0000        55.0000
    3      resource country       False     0

## Recap

In [17]:
for name in ["runs_tidy", "baseline_reference", "delta_per_attribute", "delta_aggregated", "ranking"]:
    path = os.path.join(tables_dir, f"{name}.csv")
    print(f"{name + '.csv':26s} {len(pd.read_csv(path)):6d} rows   {os.path.getsize(path) / 1024:7.1f} KB")

runs_tidy.csv               15250 rows    1650.9 KB
baseline_reference.csv        165 rows      15.3 KB
delta_per_attribute.csv      1360 rows     232.0 KB
delta_aggregated.csv          165 rows      15.2 KB
ranking.csv                    33 rows       3.0 KB
